# 01: Data Exploration

Goals:
- Verify the dataset looks sensible
- Check distributions of key variables
- Produce the log-log plot of impact vs trade size
- Establish the empirical baseline at the individual trade level

**Research question:** Does the functional form of crypto market impact match the equity model, or does the data support a different structure? Does the answer change across market regimes?

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy import stats

DATA_PATH = Path("../data/processed/impact_data.parquet")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Load

In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa

pf = pq.ParquetFile(DATA_PATH)
total_rows = pf.metadata.num_rows
print(f"Total rows: {total_rows:,}")

rng     = np.random.default_rng(42)
indices = np.sort(rng.choice(total_rows, size=5_000_000, replace=False))

batches    = []
row_offset = 0
for batch in pf.iter_batches(batch_size=1_000_000):
    batch_len = len(batch)
    mask = (indices >= row_offset) & (indices < row_offset + batch_len)
    local_indices = indices[mask] - row_offset
    if len(local_indices) > 0:
        batches.append(batch.take(local_indices))
    row_offset += batch_len
    if row_offset > indices[-1]:
        break

df = pa.Table.from_batches(batches).to_pandas().reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["timestamp_ms"], unit="ms")

print(f"Sampled rows  : {len(df):,}")
print(f"Date range    : {df['datetime'].min()} to {df['datetime'].max()}")
print(f"Columns       : {list(df.columns)}")
df.head()

## Basic sanity checks

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nSide distribution (1=buy, -1=sell):")
print(df["side"].value_counts())

print("\nKey stats:")
df[["qty", "signed_impact", "sigma_local", "volume_local"]].describe()

## Filter

Remove top 0.1% by trade size. Block trades have different impact dynamics
and would distort the regression. We keep all other trades including
zero-impact ones, these are valid observations at the individual trade level.

In [ ]:
before = len(df)
qty_cap = df["qty"].quantile(0.999)
df = df[df["qty"] <= qty_cap]
print(f"Removed {before - len(df):,} rows ({(before - len(df)) / before:.1%})")
print(f"Remaining: {len(df):,}")

## Distribution of trade sizes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["qty"], bins=100, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Trade size (BTC)")
axes[0].set_ylabel("Count")
axes[0].set_title("Trade size distribution")

axes[1].hist(np.log10(df["qty"]), bins=100, color="steelblue", edgecolor="none")
axes[1].set_xlabel("log10(Trade size)")
axes[1].set_title("Trade size distribution (log scale)")

plt.tight_layout()
plt.savefig("../outputs/figures/trade_size_dist.png", bbox_inches="tight")
plt.show()

## Volume by month

In [ ]:
monthly = df.groupby(df["datetime"].dt.to_period("M")).agg(
    trades=("qty", "count"),
    volume=("qty", "sum"),
    mean_impact=("signed_impact", "mean"),
    mean_sigma=("sigma_local", "mean"),
).reset_index()

print(monthly.to_string(index=False))

## Log-log plot: impact vs trade size

Individual trade level. We expect δ ≈ 0 here, the sqrt law emerges at the
metaorder level, not at the level of individual child orders. This plot
establishes the baseline before metaorder reconstruction.

In [ ]:
N_BINS = 50

df["qty_bin"] = pd.qcut(df["qty"], q=N_BINS, duplicates="drop")

binned = df.groupby("qty_bin", observed=True).agg(
    qty_median=("qty", "median"),
    impact_median=("signed_impact", "median"),
    count=("qty", "count"),
).reset_index()

# Keep bins with enough observations
binned = binned[binned["count"] >= 100]

log_qty = np.log(binned["qty_median"].values)
log_impact = np.log(binned["impact_median"].values)

# OLS fit in log-log space
slope, intercept, r, p, se = stats.linregress(log_qty, log_impact)

print(f"Estimated exponent δ : {slope:.4f}")
print(f"95% CI               : [{slope - 1.96*se:.4f}, {slope + 1.96*se:.4f}]")
print(f"R²                   : {r**2:.4f}")
print(f"p-value              : {p:.2e}")

## Train/test split

Split chronologically. Last 20% of data is held out as the test set.
Save both splits for use in subsequent notebooks.

In [ ]:
df = df.sort_values("timestamp_ms").reset_index(drop=True)
split_idx = int(len(df) * 0.8)

# Drop qty_bin before saving, interval dtype is not parquet-compatible
cols = [c for c in df.columns if c != "qty_bin"]
train = df[cols].iloc[:split_idx]
test  = df[cols].iloc[split_idx:]

train.to_parquet("../data/processed/train.parquet", index=False)
test.to_parquet("../data/processed/test.parquet",  index=False)

print(f"Train: {len(train):,} trades  ({train['datetime'].min().date()} to {train['datetime'].max().date()})")
print(f"Test : {len(test):,} trades  ({test['datetime'].min().date()} to {test['datetime'].max().date()})")


## Summary

δ ≈ 0 at the individual trade level is the expected result and the
correct starting point. The next step is metaorder reconstruction
(01b_metaorders.ipynb), after which we expect to recover δ ≈ 0.5.

In [ ]:
print(f"delta (slope)  : {slope:.4f}")
print(f"95% CI         : [{slope - 1.96*se:.4f}, {slope + 1.96*se:.4f}]")
print(f"R²             : {r**2:.4f}")
print(f"Square-root law (delta=0.5) is within CI: {slope - 1.96*se <= 0.5 <= slope + 1.96*se}")

---
## Metaorder reconstruction

Load the reconstructed metaorders and verify the square-root law holds.
This confirms the Maitrier et al. (2025) reconstruction worked correctly.
Parameters used: N=20 traders, power-law distribution (alpha=2), min 10 child orders.

In [ ]:
mo = pd.read_parquet("../data/processed/metaorders.parquet")

print(f"Metaorders       : {len(mo):,}")
print(f"Median n_child   : {mo['n_child'].median():.1f}")
print(f"Median Q_norm    : {mo['Q_norm'].median():.2e}")
print(f"Buy/sell split   : {(mo['sign']==1).mean():.1%} / {(mo['sign']==-1).mean():.1%}")
mo[["n_child", "Q", "Q_norm", "I"]].describe()

### Log-log plot: metaorder impact vs size

Unlike the individual trade plot in Phase 1, we expect a clear positive slope
here with δ ≈ 0.5. The dashed line is the theoretical square-root law.

In [ ]:
# Keep only positive impact metaorders for log-log plot
mo_pos = mo[mo["I"] > 0].copy()
mo_pos = mo_pos[mo_pos["Q_norm"] > 0]

# Remove outliers beyond 3 sigma in log space
log_q = np.log(mo_pos["Q_norm"].values)
log_i = np.log(mo_pos["I"].values)
mask  = (
    (np.abs(log_q - log_q.mean()) < 3 * log_q.std()) &
    (np.abs(log_i - log_i.mean()) < 3 * log_i.std())
)
log_q, log_i = log_q[mask], log_i[mask]

# Bin by Q_norm and compute median impact per bin
N_BINS = 50
mo_pos["q_bin"] = pd.qcut(mo_pos["Q_norm"], q=N_BINS, duplicates="drop")
binned = mo_pos.groupby("q_bin", observed=True).agg(
    Q_median=("Q_norm", "median"),
    I_median=("I",     "median"),
    count   =("I",     "count"),
).reset_index()
binned = binned[binned["count"] >= 50]

bq = np.log(binned["Q_median"].values)
bi = np.log(binned["I_median"].values)

slope, intercept, r, p, se = stats.linregress(bq, bi)
print(f"delta (slope) : {slope:.4f}")
print(f"95% CI        : [{slope - 1.96*se:.4f}, {slope + 1.96*se:.4f}]")
print(f"R²            : {r**2:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(bq, bi, s=20, color="steelblue", alpha=0.8, label="Binned median impact")

x_line = np.linspace(bq.min(), bq.max(), 200)
ax.plot(x_line, intercept + slope * x_line,
        color="firebrick", linewidth=1.5,
        label=f"OLS fit  δ = {slope:.3f}")
ax.plot(x_line, intercept + 0.5 * (x_line - bq.mean()) + bi.mean(),
        color="gray", linewidth=1.5, linestyle="--",
        label="Square-root law  δ = 0.5")

ax.set_xlabel("log(Q_norm), metaorder participation rate")
ax.set_ylabel("log(I), normalized impact")
ax.set_title("Metaorder Price Impact vs Size. BTC/USDT")
ax.legend()

plt.tight_layout()
plt.savefig("../outputs/figures/loglog_metaorders.png", bbox_inches="tight")
plt.show()

### Distribution of metaorder sizes and child order counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(np.log10(mo["Q_norm"].clip(lower=1e-10)), bins=80,
             color="steelblue", edgecolor="none")
axes[0].set_xlabel("log10(Q_norm)")
axes[0].set_title("Metaorder size distribution")

axes[1].hist(mo["n_child"].clip(upper=200), bins=80,
             color="steelblue", edgecolor="none")
axes[1].set_xlabel("Number of child orders")
axes[1].set_title("Metaorder length distribution")

plt.tight_layout()
plt.savefig("../outputs/figures/metaorder_distributions.png", bbox_inches="tight")
plt.show()

### Train/test split on metaorders

Chronological 80/20 split, matching the trade-level split.

In [ ]:
mo = mo.sort_values("start_ts").reset_index(drop=True)
split_idx = int(len(mo) * 0.8)

mo_train = mo.iloc[:split_idx]
mo_test  = mo.iloc[split_idx:]

mo_train.to_parquet("../data/processed/mo_train.parquet", index=False)
mo_test.to_parquet("../data/processed/mo_test.parquet",  index=False)

t_min_train = pd.to_datetime(mo_train["start_ts"].min(), unit="ms").date()
t_max_train = pd.to_datetime(mo_train["start_ts"].max(), unit="ms").date()
t_min_test  = pd.to_datetime(mo_test["start_ts"].min(),  unit="ms").date()
t_max_test  = pd.to_datetime(mo_test["start_ts"].max(),  unit="ms").date()

print(f"Train: {len(mo_train):,} metaorders  ({t_min_train} to {t_max_train})")
print(f"Test : {len(mo_test):,} metaorders  ({t_min_test} to {t_max_test})")

### Robustness check: sensitivity of delta to N traders

The reconstruction assigns synthetic trader IDs using N as a free parameter.
Here we check how sensitive the estimated delta is to that choice.
If delta is stable across a range of N, the result is not an artifact
of the parameter choice.

In [ ]:
import sys
sys.path.append("../src")
from reconstruct_metaorders import (
    build_trader_weights,
    assign_trader_ids,
    extract_metaorders,
    compute_daily_stats,
    normalize_metaorders,
)

# Load full trade data for the reconstruction
trades_full = pd.read_parquet("../data/processed/impact_data.parquet")

daily_stats = compute_daily_stats(trades_full)

N_VALUES     = [10, 20, 50, 100]
DISTRIBUTION = "power_law"
ALPHA        = 2.0
N_BINS       = 50
MIN_COUNT    = 50

sensitivity_results = []

np.random.seed(42)
for n in N_VALUES:
    weights    = build_trader_weights(n, DISTRIBUTION, ALPHA)
    trader_ids = assign_trader_ids(len(trades_full), n, weights)
    mo_n       = extract_metaorders(trades_full, trader_ids)
    mo_n       = normalize_metaorders(mo_n, daily_stats)
    mo_n       = mo_n[(mo_n["I"] > 0) & (mo_n["Q_norm"] > 0)].copy()

    log_q = np.log(mo_n["Q_norm"].values)
    log_i = np.log(mo_n["I"].values)
    mask  = (
        (np.abs(log_q - log_q.mean()) < 3 * log_q.std()) &
        (np.abs(log_i - log_i.mean()) < 3 * log_i.std())
    )
    mo_n = mo_n[mask].copy()

    mo_n["q_bin"] = pd.qcut(mo_n["Q_norm"], q=N_BINS, duplicates="drop")
    binned_n = mo_n.groupby("q_bin", observed=True).agg(
        Q_median=("Q_norm", "median"),
        I_median=("I",      "median"),
        count   =("I",      "count"),
    ).reset_index()
    binned_n = binned_n[binned_n["count"] >= MIN_COUNT]

    bq_n = np.log(binned_n["Q_median"].values)
    bi_n = np.log(binned_n["I_median"].values)

    s, _, _, _, se = stats.linregress(bq_n, bi_n)
    sensitivity_results.append({
        "N": n,
        "n_metaorders": len(mo_n),
        "median_n_child": mo_n["n_child"].median() if "n_child" in mo_n.columns else float("nan"),
        "delta": s,
        "ci_lo": s - 1.96 * se,
        "ci_hi": s + 1.96 * se,
    })
    print(f"N={n:>3}  delta={s:.4f}  CI=[{s-1.96*se:.4f}, {s+1.96*se:.4f}]  metaorders={len(mo_n):,}")

sensitivity_df = pd.DataFrame(sensitivity_results)
print()
print(sensitivity_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.errorbar(
    sensitivity_df["N"],
    sensitivity_df["delta"],
    yerr=[
        sensitivity_df["delta"] - sensitivity_df["ci_lo"],
        sensitivity_df["ci_hi"] - sensitivity_df["delta"],
    ],
    fmt="o", color="steelblue", capsize=4, linewidth=1.5,
    label="Estimated delta"
)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1.5, label="Sqrt law (delta=0.5)")
ax.set_xlabel("N traders")
ax.set_ylabel("delta")
ax.set_title("Sensitivity of delta to N traders")
ax.legend()

plt.tight_layout()
plt.savefig("../outputs/figures/delta_sensitivity_N.png", bbox_inches="tight")
plt.show()

### Summary

Record δ from the metaorder log-log plot. This is the key result of Phase 1
and the number the MLP will be compared against.

In [ ]:
print(f"N traders        : 20 (power-law, alpha=2)")
print(f"Metaorders       : {len(mo):,}")
print(f"Median n_child   : {mo['n_child'].median():.1f}")
print(f"delta (slope)    : {slope:.4f}")
print(f"95% CI           : [{slope - 1.96*se:.4f}, {slope + 1.96*se:.4f}]")
print(f"R² (binned)      : {r**2:.4f}")